# **Step 1: Extract relevant content features.**

In [3]:
# importing basic libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# loading the cleaned dataset
df = pd.read_csv('/content/cleaned_netflix.csv')

# checking if data loaded properly
print(df.shape)
df.head()

# combining title, director, genre, and country into one column
df['combined'] = df['title'] + ' ' + df['director'] + ' ' + df['listed_in'] + ' ' + df['country']

# checking the new column
print(df[['title', 'combined']].head())

(8790, 13)
                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                            combined  
0  Dick Johnson Is Dead Kirsten Johnson Documenta...  
1  Ganglands Julien Leclercq Crime TV Shows, Inte...  
2  Midnight Mass Mike Flanagan TV Dramas, TV Horr...  
3  Confessions of an Invisible Girl Bruno Garotti...  
4  Sankofa Haile Gerima Dramas, Independent Movie...  


# **Step 2: Perform text preprocessing.**

In [4]:
# simple function to clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

# applying cleaning on combined column
df['clean_text'] = df['combined'].apply(clean_text)

# checking cleaned text
print(df[['title', 'clean_text']].head())

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                          clean_text  
0  dick johnson is dead kirsten johnson documenta...  
1  ganglands julien leclercq crime tv shows inter...  
2  midnight mass mike flanagan tv dramas tv horro...  
3  confessions of an invisible girl bruno garotti...  
4  sankofa haile gerima dramas independent movies...  


# **Step 3: Calculate content similarity**

In [5]:
# converting text into numbers using TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

# checking matrix shape
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# calculating similarity between all titles
similarity_score = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Similarity matrix created successfully!")
print("Shape:", similarity_score.shape)

TF-IDF matrix shape: (8790, 14509)
Similarity matrix created successfully!
Shape: (8790, 8790)


# **Step 4: Generate recommendations for selected titles.**

In [10]:
# function to get top 5 similar titles
def recommend(title_name):
    # finding the index of the given title
    index = df[df['title'] == title_name].index[0]

    # getting similarity scores for that title
    scores = list(enumerate(similarity_score[index]))

    # sorting scores from high to low
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # taking top 5 (excluding the first one which is the title itself)
    top_5 = sorted_scores[1:6]

    # getting the titles from indices
    movie_list = [df['title'].iloc[i[0]] for i in top_5]

    return movie_list

    # testing the function with real titles from dataset
print("Recommendations for 'Ganglands':")
print(recommend('Ganglands'))


Recommendations for 'Ganglands':
['Sentinelle', 'Crime Time', 'Earth and Blood', 'Lupin', 'H']


# **Step 5: Evaluate recommendation quality.**

In [8]:
# checking recommendations for multiple real titles
print("Testing recommendation quality for different titles:\n")

print("1. Recommendations for 'Ganglands':")
print(recommend('Ganglands'))

print("\n2. Recommendations for 'Midnight Mass':")
print(recommend('Midnight Mass'))

print("\n3. Recommendations for 'Sankofa':")
print(recommend('Sankofa'))

print("\n4. Recommendations for 'Dick Johnson Is Dead':")
print(recommend('Dick Johnson Is Dead'))

Testing recommendation quality for different titles:

1. Recommendations for 'Ganglands':
['Sentinelle', 'Crime Time', 'Earth and Blood', 'Lupin', 'H']

2. Recommendations for 'Midnight Mass':
['Before I Wake', 'Hush', 'Somewhere Between', "Gerald's Game", 'American Horror Story']

3. Recommendations for 'Sankofa':
['Residue', 'I Am', 'The Get Down', 'Only', 'Brain on Fire']

4. Recommendations for 'Dick Johnson Is Dead':
['Triple Threat', 'S.W.A.T.', 'Home', 'Brick', 'Avengement']
